# Lab 10: Praxisprojekt und Interpretation

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Datenordner finden: Notebook liegt in labs/ oder loesungen/, die Daten in data/
DATA = next(p for p in [Path("data"), Path("../data"), Path("../../data")] if p.exists())
print("Datenordner:", DATA)

Dieses Lab gehört zu **Teil 10: Praxisbeispiele und Interpretation von Modellen**.

## Lernziele

- Sie wählen Merkmale so, dass keine Spalte die Antwort schon enthält.
- Sie bewerten ein Modell bei stark ungleichen Klassen mit Recall und Precision und legen eine Schwelle fest.
- Sie vergleichen `feature_importances_` mit `permutation_importance` und lesen einen Entscheidungsbaum.
- Sie lesen die Koeffizienten eines linearen Modells nach dem Skalieren und erkennen im Residuenplot eine gedeckelte Zielgröße.
- Sie führen ein eigenes Mini-Projekt von der Frage bis zu drei Sätzen für die Fachabteilung durch.

Aufbau: **Teil A** (Blöcke 1 bis 3) ist geführt, mit Kontrollergebnissen zu jeder Aufgabe. **Teil B** (Block 4) ist Ihr eigenes Projekt auf einem Datensatz nach Wahl. Die Zahlen gelten für `random_state=1`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (train_test_split, cross_val_score, cross_validate,
                                     cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.inspection import permutation_importance
from sklearn.metrics import (classification_report, confusion_matrix, recall_score,
                             precision_score, mean_squared_error, r2_score)

df_m = pd.read_csv(DATA / "ai4i2020.csv")
df_m.head()

# Teil A: geführte Beispiele

## Block 1: Maschinenausfall vorhersagen

Der Datensatz beschreibt 10 000 Fertigungsläufe. Die Zielgröße `Machine failure` sagt, ob die Maschine ausgefallen ist.

1. Sehen Sie sich Form und Zielgröße an: Wie viele Ausfälle gibt es, und welche Accuracy erreicht ein Modell, das immer „kein Ausfall" sagt? Erwartet: `(10000, 14)`, 339 Ausfälle, Anteil 0.034, also Accuracy 0.966 ohne jedes Lernen.
2. **Verständnisaufgabe:** Wählen Sie die Merkmale. Prüffrage für jede Spalte: „Kenne ich diesen Wert, bevor der Ausfall eintritt?" Die Spalten `TWF`, `HDF`, `PWF`, `OSF`, `RNF` nennen die Art des Ausfalls und stehen erst fest, wenn er passiert ist. `UDI` und `Product ID` sind Kennungen. Alle sieben gehören nicht in `X`. Teilen Sie danach mit `test_size=0.2`, `random_state=1`, `stratify=y` und bauen Sie `prep_m` (Zahlen skalieren, `Type` one-hot). Erwartet: 6 Merkmale, `(8000, 6)` und `(2000, 6)`, 68 Ausfälle im Testset.
3. Bewerten Sie mit der vorbereiteten Funktion `bewerte` die Basislinie `DummyClassifier(strategy="most_frequent")` und einen `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced", random_state=1)`. Erwartet: Basislinie Accuracy 0.97, Recall 0.0, Precision 0.0. Random Forest Accuracy 0.96, Recall 0.85, Precision 0.43. Nach Accuracy läge die nutzlose Basislinie sogar vorn.
4. Legen Sie die Schwelle fest. Berechnen Sie mit `cross_val_predict(..., method="predict_proba")` die Wahrscheinlichkeiten auf den Trainingsdaten und daraus Recall und Precision für die Schwellen 0.7, 0.5, 0.3 und 0.2. Werten Sie danach das Testset **einmal** mit Schwelle 0.3 aus. Erwartet: in der Cross-Validation bei Schwelle 0.7 Recall 0.66 und Precision 0.60, bei Schwelle 0.3 Recall 0.92 und Precision 0.27. Testset mit 0.3: Matrix `[[1792 140] [3 65]]`, also 65 von 68 Ausfällen gefunden bei 140 Fehlalarmen auf 2000 Zeilen. Welche Schwelle richtig ist, entscheidet die Fachseite: Was kostet ein übersehener Ausfall, was eine unnötige Wartung?

In [ ]:
# Aufgabe 1: Form und Zielgröße ansehen
# Tipp: shape, value_counts(), mean()
anteil = ...
print(anteil)

In [ ]:
# Aufgabe 2: Merkmale wählen, ohne die Antwort mitzugeben
print(list(df_m.columns))
# Tragen Sie nur Spalten ein, die VOR dem Ausfall bekannt sind
num_cols_m = ...
cat_cols_m = ...
# Xm = df_m[num_cols_m + cat_cols_m]
# ym = df_m["Machine failure"]
# Xm_train, Xm_test, ym_train, ym_test = train_test_split(...)
# prep_m = ColumnTransformer([("num", StandardScaler(), num_cols_m),
#                             ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_m)])
# print(Xm_train.shape, Xm_test.shape, "| Ausfälle im Test:", ym_test.sum())

In [ ]:
def bewerte(modell):
    """Cross-Validation der Pipeline aus prep_m und modell auf den Trainingsdaten, drei Kennzahlen."""
    pipe = Pipeline([("prep", prep_m), ("modell", modell)])
    cv = cross_validate(pipe, Xm_train, ym_train, cv=5,
                        scoring=["accuracy", "recall", "precision"])
    return {k[5:]: round(float(v.mean()), 2) for k, v in cv.items() if k.startswith("test_")}

In [ ]:
# Aufgabe 3: Basislinie gegen Random Forest mit class_weight="balanced"
# Tipp: bewerte(DummyClassifier(strategy="most_frequent")), Ergebnisse in ein DataFrame
wald = ...
tabelle_m = ...
tabelle_m

In [ ]:
# Aufgabe 4: Schwelle über Cross-Validation wählen, danach Testset einmal mit 0.3
# rf_m = Pipeline([("prep", prep_m), ("modell", wald)])
# proba_cv = cross_val_predict(rf_m, Xm_train, ym_train, cv=5, method="predict_proba")[:, 1]
# Schleife über [0.7, 0.5, 0.3, 0.2]: recall_score(ym_train, proba_cv >= s), precision_score(...)
# Danach: rf_m.fit(...), y_pred_m = (rf_m.predict_proba(Xm_test)[:, 1] >= 0.3).astype(int)
rf_m = ...

## Block 2: Was hat das Modell gelernt?

Für die Interpretation wechseln Sie zu Titanic, weil Sie die Spalten kennen. Die folgende Zelle lädt die Daten, fügt eine Spalte `zufall` aus reinen Zufallszahlen hinzu und baut die bekannte Aufbereitung. Die Zufallsspalte hat mit dem Überleben nichts zu tun. Ein gutes Verfahren muss sie ans Ende der Rangfolge setzen.

1. Trainieren Sie die Pipeline `rf_t` (Aufbereitung `prep_t`, `RandomForestClassifier(n_estimators=200, random_state=1)`) und geben Sie `feature_importances_` mit Namen absteigend sortiert aus. Erwartet: `num__zufall` landet mit 0.189 auf Platz 1 von 13, vor `num__Fare` (0.172), `num__Age` (0.170) und beiden `Sex`-Spalten. Die Accuracy auf den Trainingsdaten ist 1.0: Der Wald lernt auswendig und nutzt dafür auch die Zufallszahlen.
2. Berechnen Sie `permutation_importance` auf den **Testdaten** (`n_repeats=20`, `random_state=1`) und sortieren Sie wieder. Erwartet: `Sex` vorn mit 0.218, danach `Pclass` mit 0.095. `zufall` liegt bei 0.005 mit Streuung 0.009 und ist damit von null nicht zu unterscheiden.
3. Zeichnen Sie mit `plot_tree` einen Entscheidungsbaum der Tiefe 3 (`min_samples_leaf=20`) mit Merkmalsnamen, ohne die Zufallsspalte. Lassen Sie im Zahlenzweig die Skalierung weg, damit die Schwellen in Originaleinheiten lesbar sind. Erwartet: Die Wurzel fragt nach dem Geschlecht, der Baum hat 8 Blätter. Formulieren Sie eine Regel in einem Satz.

In [ ]:
df_t = pd.read_csv(DATA / "titanic.csv")
df_t["zufall"] = np.random.default_rng(42).normal(size=len(df_t))   # reine Zufallszahlen

num_cols_t = ["Age", "Fare", "SibSp", "Parch"]
cat_cols_t = ["Pclass", "Sex", "Embarked"]
Xt = df_t[num_cols_t + ["zufall"] + cat_cols_t]
yt = df_t["Survived"]
Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    Xt, yt, test_size=0.2, random_state=1, stratify=yt)

numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                             ("encoder", OneHotEncoder(handle_unknown="ignore"))])
prep_t = ColumnTransformer([
    ("num", numeric_pipe, num_cols_t + ["zufall"]),
    ("cat", categorical_pipe, cat_cols_t),
])
print(Xt_train.shape)

In [ ]:
# Aufgabe 1: feature_importances_ des Random Forest
# Tipp: Werte aus rf_t.named_steps["modell"].feature_importances_,
#       Namen aus rf_t.named_steps["prep"].get_feature_names_out(), beides in eine pd.Series
rf_t = ...
wichtigkeit = ...
wichtigkeit

In [ ]:
# Aufgabe 2: permutation_importance auf den Testdaten
# Tipp: permutation_importance(rf_t, Xt_test, yt_test, n_repeats=20, random_state=1)
#       result.importances_mean und result.importances_std, Index Xt_test.columns
perm = ...
perm

In [ ]:
# Aufgabe 3: Entscheidungsbaum der Tiefe 3 zeichnen
# Tipp: eigene Aufbereitung prep_baum ohne zufall, im Zahlenzweig nur SimpleImputer(strategy="median")
#       plot_tree(baum.named_steps["modell"], feature_names=..., class_names=["gestorben", "überlebt"],
#                 filled=True, rounded=True, impurity=False, proportion=True, fontsize=8, ax=ax)
baum = ...
# Regel in einem Satz:

## Block 3: Regression auf California Housing

Zielgröße ist `MedHouseVal`, der Median-Hauswert eines Bezirks in 100.000 USD. Alle acht Merkmale sind Zahlen ohne Lücken.

1. Teilen Sie (`test_size=0.2`, `random_state=1`, kein `stratify`) und bauen Sie die Pipeline `reg` aus `StandardScaler` und `LinearRegression()`. Bestimmen Sie den RMSE per Cross-Validation (`scoring="neg_root_mean_squared_error"`) und auf dem Testset, dazu R². Erwartet: RMSE Cross-Validation 0.727, RMSE Test 0.727, R² 0.597.
2. Lesen Sie die Koeffizienten als Tabelle, sortiert nach Betrag. Erwartet: `Latitude` -0.91, `Longitude` -0.89 und `MedInc` 0.83 liegen vorn, `Population` ist fast null (-0.004).
3. Zeichnen Sie den Residuenplot (Vorhersage gegen Residuum) und suchen Sie die Deckelung der Zielgröße. Zählen Sie die Zeilen mit `MedHouseVal >= 5`. Erwartet: größter Wert 5.00001, 992 Bezirke (4.8 Prozent) liegen an der Deckelung. Im Plot bilden sie eine schräge obere Kante.

In [ ]:
# Aufgabe 1: Pipeline aus Scaler und linearer Regression, RMSE und R²
df_h = pd.read_csv(DATA / "california_housing.csv")
# Xh = df_h.drop(columns="MedHouseVal"), yh = df_h["MedHouseVal"], danach train_test_split
# reg = Pipeline([("scaler", ...), ("modell", LinearRegression())])
# Tipp: -cross_val_score(..., scoring="neg_root_mean_squared_error") und np.sqrt(mean_squared_error(...))
reg = ...

In [ ]:
# Aufgabe 2: Koeffizienten als sortierte Tabelle
# Tipp: reg.named_steps["modell"].coef_, Index Xh_train.columns, sortieren nach Betrag
koef = ...
koef

In [ ]:
# Aufgabe 3: Residuenplot und Deckelung
# residuen = yh_test - yh_pred
# Tipp: ax.scatter(yh_pred, residuen, s=6, alpha=0.3), ax.axhline(0, ...)
# Deckelung: yh.max() und (yh >= 5).sum()
gedeckelt = ...
print(gedeckelt)

# Teil B: Ihr eigenes Mini-Projekt

## Block 4: Von der Frage bis zum Ergebnis für die Fachabteilung

Wählen Sie einen Datensatz und eine Frage:

| Datei | mögliche Frage | Art |
|---|---|---|
| `titanic.csv` | Wer überlebt? | Klassifikation |
| `california_housing.csv` | Wie hoch ist der Hauswert eines Bezirks? | Regression |
| `winequality-red.csv` (Trennzeichen `;`) | Ist ein Wein gut (`quality >= 6`)? Oder: Welche Note bekommt er? | Klassifikation oder Regression |
| `versicherte.csv` | Welche Gruppen von Versicherten gibt es? | Segmentierung |

Arbeiten Sie die zehn Schritte der Reihe nach ab. Jede Zelle unten ist ein Schritt. Bei einer Segmentierung gibt es keine Zielgröße: Dort entfallen Split, Basislinie und Testauswertung, stattdessen berechnen und benennen Sie ein Segmentprofil.

1. **Frage formulieren:** Welche Entscheidung soll das Modell unterstützen? Welche Kennzahl passt dazu?
2. **Daten ansehen:** Form, Datentypen, fehlende Werte, Duplikate, Verteilung der Zielgröße.
3. **Zielgröße und Merkmale:** Kennungen weglassen. Prüffrage für jede Spalte: Kenne ich den Wert vor dem Ereignis?
4. **Split:** `random_state=1`, bei Klassifikation `stratify=y`. Das Testset legen Sie zur Seite.
5. **Basislinie:** `DummyClassifier` oder `DummyRegressor`. Diese Zahl muss Ihr Modell schlagen.
6. **Pipeline:** Aufbereitung und Modell in einem Objekt, am besten zwei Kandidaten.
7. **Cross-Validation:** Kandidaten auf den Trainingsdaten vergleichen, einen auswählen.
8. **Testauswertung:** genau einmal, mit Konfusionsmatrix oder Residuenplot.
9. **Interpretation:** `permutation_importance` auf den Testdaten, Richtung aus Koeffizienten.
10. **Drei Sätze für die Fachabteilung:** Wie gut ist es? Worauf stützt es sich? Wo darf man ihm nicht trauen?

Kontrollergebnisse hängen von Ihrer Wahl ab. Zur Orientierung die Werte der Musterlösung (Wein, Klassifikation „gut" ab `quality >= 6`, 1599 Zeilen, nach Entfernen von 240 doppelten Zeilen 1359, Anteil „gut" 0.529): Basislinie Accuracy 0.529, logistische Regression in der Cross-Validation 0.727, Random Forest 0.733 (Streuung 0.041 und 0.050), Testset der logistischen Regression Accuracy 0.75 mit Matrix `[[104 24] [44 100]]`, wichtigste Merkmale `alcohol`, `volatile acidity`, `sulphates`.

In [ ]:
# Schritt 1: Frage formulieren
# Welche Entscheidung soll das Modell unterstützen? Klassifikation, Regression oder Segmentierung?
frage = "..."
kennzahl = "..."
print(frage, "| Kennzahl:", kennzahl)

In [ ]:
# Schritt 2: Daten ansehen
# Tipp: pd.read_csv(DATA / "...", sep=";" nur bei der Weindatei), shape, dtypes, isna().sum(), duplicated().sum()
daten = ...

In [ ]:
# Schritt 3: Zielgröße und Merkmale festlegen
X = ...
y = ...

In [ ]:
# Schritt 4: Split (Testset danach zur Seite legen)
# X_train, X_test, y_train, y_test = train_test_split(...)

In [ ]:
# Schritt 5: Basislinie
# Tipp: cross_val_score(DummyClassifier(strategy="most_frequent"), X_train, y_train, cv=5)
basis = ...

In [ ]:
# Schritt 6: Pipeline bauen (am besten zwei Kandidaten)
kandidaten = {
    # "Name": Pipeline([...]),
}

In [ ]:
# Schritt 7: Cross-Validation auf den Trainingsdaten, Kandidaten vergleichen
# Tipp: cross_validate(pipe, X_train, y_train, cv=5, scoring=[...]) in einer Schleife über kandidaten
vergleich = ...
vergleich

In [ ]:
# Schritt 8: Testauswertung, genau einmal
final = ...

In [ ]:
# Schritt 9: Interpretation
# Tipp: permutation_importance(final, X_test, y_test, n_repeats=20, random_state=1), Balkendiagramm

In [ ]:
# Schritt 10: drei Sätze für die Fachabteilung
ergebnis = """
1. Wie gut ist es? ...
2. Worauf stützt es sich? ...
3. Wo darf man ihm nicht trauen? ...
"""
print(ergebnis)

## Zusatzaufgaben

1. **Das Leck absichtlich einbauen:** Trainieren Sie den Random Forest aus Block 1 zusätzlich mit den Spalten `TWF`, `HDF`, `PWF`, `OSF`, `RNF` und bewerten Sie per Cross-Validation. Erwartet: Accuracy 0.998, Recall 0.96, Precision 0.996. Erklären Sie in einem Kommentar, warum das Ergebnis wertlos ist.
2. **Permutation Importance passend zur Frage:** Berechnen Sie für `rf_m` die Permutation Importance auf den Testdaten mit `scoring="recall"` (`n_repeats=10`). Erwartet: `Torque [Nm]` vorn (0.39), danach `Rotational speed [rpm]` (0.28) und `Tool wear [min]` (0.22). `Type` (0.03) und `Process temperature [K]` (0.00) liegen fast bei null.
3. **Deckelung herausnehmen:** Entfernen Sie alle Zeilen mit `MedHouseVal >= 5`, teilen und trainieren Sie die Pipeline neu und vergleichen Sie den RMSE auf dem Testset. Erwartet: 19648 Zeilen, RMSE 0.624 statt 0.727. Beachten Sie: Das Testset ist jetzt ein anderes, die Aussage des Modells gilt nur noch für Bezirke unter der Deckelung.

In [ ]:
# Zusatz 1: Modell mit den Ausfallarten als Merkmal
leck_spalten = ["TWF", "HDF", "PWF", "OSF", "RNF"]
# Tipp: eigener ColumnTransformer, der leck_spalten im Zahlenzweig mitnimmt, danach cross_validate
# Warum ist das Ergebnis wertlos?

In [ ]:
# Zusatz 2: Permutation Importance mit scoring="recall" auf den Maschinendaten
perm_m = ...
perm_m

In [ ]:
# Zusatz 3: Zeilen an der Deckelung entfernen und neu trainieren
df_h2 = ...

## Was Sie mitnehmen

- Vor dem Modell steht die Spaltenwahl: Eine Spalte, die erst nach dem Ereignis feststeht, ist ein Datenleck. Bei seltenen Ereignissen zählen Recall, Precision und eine bewusst gewählte Schwelle, nicht die Accuracy.
- `feature_importances_` stammt aus den Trainingsdaten und bevorzugt Spalten mit vielen Werten. `permutation_importance` auf Testdaten setzt eine Zufallsspalte dorthin, wo sie hingehört. Beide sagen nichts über Ursachen.
- Ein Ergebnis ist erst fertig, wenn Basislinie, Fehlerbild in Fällen, wichtigste Merkmale und Grenzen in wenigen Sätzen für die Fachabteilung stehen.